In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# yolo_dataset_v2의 라벨을 Faster R-CNN 포맷(x1,y1,x2,y2 절대좌표)으로 변환
import os, json
from PIL import Image

dataset_path = "/content/drive/MyDrive/data/초급_프로젝트/yolo_dataset_v2"
out = "/content/frcnn_dataset"

for split in ['train', 'val']:
    os.makedirs(f"{out}/{split}/images", exist_ok=True)
    os.makedirs(f"{out}/{split}/annotations", exist_ok=True)

for split in ['train', 'val']:
    img_dir = f"{dataset_path}/{split}/images"
    lbl_dir = f"{dataset_path}/{split}/labels"

    records = []
    for img_fname in sorted(os.listdir(img_dir)):
        stem = os.path.splitext(img_fname)[0]
        img_path = os.path.join(img_dir, img_fname)
        lbl_path = os.path.join(lbl_dir, stem + '.txt')

        img = Image.open(img_path)
        W, H = img.size

        boxes = []
        with open(lbl_path, 'r') as f:
            for line in f.read().strip().splitlines():
                _, cx, cy, nw, nh = map(float, line.split())
                x1 = (cx - nw / 2) * W
                y1 = (cy - nh / 2) * H
                x2 = (cx + nw / 2) * W
                y2 = (cy + nh / 2) * H
                boxes.append([x1, y1, x2, y2])

        records.append({
            "file_name": img_fname,
            "width": W,
            "height": H,
            "boxes": boxes,
            "labels": [1] * len(boxes)  # 0은 background, 1은 pill
        })

    with open(f"{out}/{split}/annotations/data.json", 'w') as f:
        json.dump(records, f, indent=2)

    print(f"{split}: {len(records)}개 변환 완료")

train: 1740개 변환 완료
val: 231개 변환 완료


In [6]:
import shutil

dst = "/content/drive/MyDrive/data/초급_프로젝트/frcnn_dataset_v1"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree("/content/frcnn_dataset", dst)

print(f"Google Drive 저장 완료: {dst}")
for split in ['train', 'val']:
    n = len(os.listdir(f"{dst}/{split}/annotations"))
    print(f"  {split}/annotations: {n}개")

Google Drive 저장 완료: /content/drive/MyDrive/data/초급_프로젝트/frcnn_dataset_v1
  train/annotations: 1개
  val/annotations: 1개


In [1]:
import shutil, os

for split in ['train', 'val']:
    src = f"/content/drive/MyDrive/data/초급_프로젝트/yolo_dataset_v2/{split}/images"
    dst = f"/content/drive/MyDrive/data/초급_프로젝트/frcnn_dataset_v1/{split}/images"
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    print(f"{split}/images: {len(os.listdir(dst))}개 복사 완료")

train/images: 1740개 복사 완료
val/images: 231개 복사 완료
